# Random Forest and XGBoost Model Testing
This notebook builds two models for the irrigation need competition using a fast single train/test evaluation:

- `RandomForestClassifier` fit on a train split and evaluated on a held-out test split
- `XGBClassifier` fit on the same train/test split without cross-validation

In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from xgboost import XGBClassifier

train_path = '../Kaggle/train.csv'
train = pd.read_csv(train_path)

# Prepare features and target
feature_cols = [c for c in train.columns if c not in ['id', 'Irrigation_Need']]
X = pd.get_dummies(train[feature_cols], drop_first=True)
y = train['Irrigation_Need']

# Encode string classes for XGBoost compatibility
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)
label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))
print('Label mapping:', label_mapping)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print('X shape:', X.shape)
print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)
print('Target distribution:')
print(pd.Series(y).value_counts())

Label mapping: {'High': np.int64(0), 'Low': np.int64(1), 'Medium': np.int64(2)}
X shape: (630000, 35)
Train shape: (504000, 35)
Test shape: (126000, 35)
Target distribution:
1    369917
2    239074
0     21009
Name: count, dtype: int64


In [3]:
# Train a fast RandomForest model
rf = RandomForestClassifier(
    random_state=42,
    n_estimators=100,
    max_depth=20,
    n_jobs=-1
)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)
print('RandomForest test classification report:')
print(classification_report(y_test, rf_preds))

RandomForest test classification report:
              precision    recall  f1-score   support

        High       0.97      0.90      0.93      4202
         Low       0.99      1.00      0.99     73983
      Medium       0.98      0.98      0.98     47815

    accuracy                           0.99    126000
   macro avg       0.98      0.96      0.97    126000
weighted avg       0.99      0.99      0.99    126000



In [7]:
# Train a fast XGBoost model
xgb = XGBClassifier(
    random_state=42,
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='mlogloss',
    n_jobs=-1
)
xgb.fit(X_train, y_train)
xgb_preds = xgb.predict(X_test)
print('\nXGBoost test classification report:')
print(classification_report(y_test, xgb_preds))

c:\Users\tyler\anaconda3\Lib\site-packages\xgboost\training.py:199: UserWarning: [11:09:00] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



XGBoost test classification report:
              precision    recall  f1-score   support

           0       0.96      0.92      0.94      4202
           1       0.99      0.99      0.99     73983
           2       0.98      0.98      0.98     47815

    accuracy                           0.99    126000
   macro avg       0.98      0.96      0.97    126000
weighted avg       0.98      0.99      0.98    126000

